# Guidelines for Prompting
# 提示指南
In this lesson, you'll practice two prompting principles and their related tactics in order to write effective prompts for large language models.
在本课程中，您将练习两个提示原则及其相关策略，以便为大型语言模型编写有效的提示。

## Setup
## 设置
#### Load the API key and relevant Python libaries.
#### 加载 API 密钥和相关 Python 库。

In this course, we've provided some code that loads the OpenAI API key for you.
在本课程中，我们提供了一些为您加载 OpenAI API 密钥的代码。

In [2]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

load_dotenv(find_dotenv())

# 创建客户端（指向通义千问 API）
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),  # 从环境变量读取
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

#### helper function
#### 辅助函数
Throughout this course, we will use OpenAI's `gpt-3.5-turbo` model and the [chat completions endpoint](https://platform.openai.com/docs/guides/chat). 
在本课程中，我们将使用 OpenAI 的“gpt-3.5-turbo”模型和[聊天完成端点](https://platform.openai.com/docs/guides/chat)。

This helper function will make it easier to use prompts and look at the generated outputs.  
此辅助函数将使使用提示和查看生成的输出变得更加容易。
**Note**: In June 2023, OpenAI updated gpt-3.5-turbo. The results you see in the notebook may be slightly different than those in the video. Some of the prompts have also been slightly modified to product the desired results.
**注**：2023 年 6 月，OpenAI 更新了 gpt-3.5-turbo。您在笔记本中看到的结果可能与视频中的结果略有不同。一些提示也进行了轻微修改，以产生所需的结果。

In [3]:
def get_completion(prompt, model="qwen-max"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

**Note:** This and all other lab notebooks of this course use OpenAI library version `0.27.0`. 
**注意：** 本课程以及本课程的所有其他实验笔记本均使用 OpenAI 库版本“0.27.0”。

In order to use the OpenAI library version `1.0.0`, here is the code that you would use instead for the `get_completion` function:
为了使用 OpenAI 库版本“1.0.0”，您可以使用以下代码来代替“get_completion”函数：

```python
client = openai.OpenAI()

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content
```

## Prompting Principles
## 提示原则
- **Principle 1: Write clear and specific instructions**
- **原则 1：写出清晰具体的说明**
- **Principle 2: Give the model time to “think”**
- **原则 2：给模型时间“思考”**

### Tactics
### 策略

#### Tactic 1: Use delimiters to clearly indicate distinct parts of the input
#### 策略 1：使用分隔符清楚地指示输入的不同部分
- Delimiters can be anything like: ```, """, < >, `<tag> </tag>`, `:`
- 分隔符可以是任何类似：```、"""、< >、`<tag> </tag>`、`:`

In [5]:
text = f"""
You should express what you want a model to do by \ 
providing instructions that are as clear and \ 
specific as you can possibly make them. \ 
This will guide the model towards the desired output, \ 
and reduce the chances of receiving irrelevant \ 
or incorrect responses. Don't confuse writing a \ 
clear prompt with writing a short prompt. \ 
In many cases, longer prompts provide more clarity \ 
and context for the model, which can lead to \ 
more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks \ 
into a single sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)

Clear and specific instructions, which may be longer to provide adequate context, are essential for guiding a model to produce the desired, relevant output.


#### Tactic 2: Ask for a structured output
#### 策略 2：要求结构化输出
- JSON, HTML
- JSON、HTML

下方提示词中文翻译
生成三本虚构书名及其作者和类型的列表。
以 JSON 格式提供以下键：
book_id、标题、作者、类型。

In [6]:
prompt = f"""
Generate a list of three made-up book titles along \ 
with their authors and genres. 
Provide them in JSON format with the following keys: 
book_id, title, author, genre.
"""
response = get_completion(prompt)
print(response)

```json
[
    {
        "book_id": 1,
        "title": "Whispers of the Forgotten Isles",
        "author": "Lila Marson",
        "genre": "Fantasy"
    },
    {
        "book_id": 2,
        "title": "The Last Echo in Time",
        "author": "Ethan Voss",
        "genre": "Science Fiction"
    },
    {
        "book_id": 3,
        "title": "Shadows Over Thornfield",
        "author": "Sophia Blackwood",
        "genre": "Mystery"
    }
]
```


#### Tactic 3: Ask the model to check whether conditions are satisfied
#### 策略 3：要求模型检查条件是否满足

下方提示词中文翻译
您将获得由三重引号分隔的文本。
如果它包含一系列指令，请按照以下格式重新编写这些指令：

步骤 1 - ...
步骤 2 - …
…
步骤 N - …

如果文本不包含一系列指令，则只需写下“未提供步骤。”

In [7]:
text_1 = f"""
Making a cup of tea is easy! First, you need to get some \ 
water boiling. While that's happening, \ 
grab a cup and put a tea bag in it. Once the water is \ 
hot enough, just pour it over the tea bag. \ 
Let it sit for a bit so the tea can steep. After a \ 
few minutes, take out the tea bag. If you \ 
like, you can add some sugar or milk to taste. \ 
And that's it! You've got yourself a delicious \ 
cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Get some water boiling.
Step 2 - Grab a cup and put a tea bag in it.
Step 3 - Once the water is hot enough, pour it over the tea bag.
Step 4 - Let the tea sit for a bit to steep.
Step 5 - After a few minutes, take out the tea bag.
Step 6 - If you like, add some sugar or milk to taste.


In [8]:
text_2 = f"""
The sun is shining brightly today, and the birds are \
singing. It's a beautiful day to go for a \ 
walk in the park. The flowers are blooming, and the \ 
trees are swaying gently in the breeze. People \ 
are out and about, enjoying the lovely weather. \ 
Some are having picnics, while others are playing \ 
games or simply relaxing on the grass. It's a \ 
perfect day to spend time outdoors and appreciate the \ 
beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.


#### Tactic 4: "Few-shot" prompting
#### 策略四：“少击”提示

下方提示词中文翻译
你的任务是以一致的方式回答。

<grandparent>：冲刷最深山谷的河流，源自一眼不起眼的泉眼；最宏大的交响乐，源于一个音符；最精致的挂毯，始于一根孤独的线。

<child>：教我如何保持耐心。

<child>：教我如何增强韧性。

In [9]:
prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest \ 
valley flows from a modest spring; the \ 
grandest symphony originates from a single note; \ 
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)

<grandparent>: Just as the willow bends but never breaks in the fiercest of storms, so too must we learn to bend with life's challenges. The oak that stands tall and rigid may crack under pressure, but the willow, with its gentle strength, endures. Resilience is not about being unyielding; it's about having the flexibility to withstand and recover from adversity. Remember, after every storm, there comes a calm, and it's in those moments of calm that we grow stronger, wiser, and more prepared for the next challenge.


### Principle 2: Give the model time to “think” 
### 原则 2：给模型时间“思考”

#### Tactic 1: Specify the steps required to complete a task
#### 策略 1：指定完成任务所需的步骤

下方提示词中文翻译
执行以下操作：
1 - 用一句话总结以下由三个反引号分隔的文本。
2 - 将摘要翻译成法语。
3 - 列出法语摘要中的每个名称。
4 - 输出一个包含以下键的 json 对象：french_summary, num_names.

使用换行符分隔答案。

文本：

In [10]:
text = f"""
In a charming village, siblings Jack and Jill set out on \ 
a quest to fetch water from a hilltop \ 
well. As they climbed, singing joyfully, misfortune \ 
struck—Jack tripped on a stone and tumbled \ 
down the hill, with Jill following suit. \ 
Though slightly battered, the pair returned home to \ 
comforting embraces. Despite the mishap, \ 
their adventurous spirits remained undimmed, and they \ 
continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple \
backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following \
keys: french_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
Jack and Jill, after a minor accident while fetching water, returned home with their adventurous spirits still high.

Jack et Jill, après un petit accident lors de la récupération d'eau, sont rentrés chez eux avec leurs esprits aventureux toujours aussi vifs.

- Jack
- Jill

{
  "french_summary": "Jack et Jill, après un petit accident lors de la récupération d'eau, sont rentrés chez eux avec leurs esprits aventureux toujours aussi vifs.",
  "num_names": 2
}


#### Ask for output in a specified format
#### 要求以指定格式输出

In [11]:
prompt_2 = f"""
Your task is to perform the following actions: 
1 - Summarize the following text delimited by 
  <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the 
  following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Summary: Jack and Jill, despite falling down a hill during their quest for water, returned home with their adventurous spirits undimmed.
Translation: Malgré leur chute pendant leur quête d'eau, Jack et Jill sont rentrés chez eux avec un esprit aventureux intact.
Names: Jack, Jill
Output JSON: {"french_summary": "Malgré leur chute pendant leur quête d'eau, Jack et Jill sont rentrés chez eux avec un esprit aventureux intact.", "num_names": 2}


#### Tactic 2: Instruct the model to work out its own solution before rushing to a conclusion
下方提示词中文翻译
确定学生的解决方案是否正确。

问题：
我正在建造一个太阳能装置，需要财务方面的帮助。
- 土地成本 100 美元/平方英尺
- 我可以以每平方英尺 250 美元的价格购买太阳能电池板
- 我协商了一份维护合同，每年固定费用为 10 万美元，另外每平方英尺 10 美元
第一年运营的总成本与平方英尺数的关系是多少？

学生的解决方案：
令 x 为安装尺寸（以平方英尺为单位）。
成本：
1.土地成本：100 倍
2.太阳能电池板成本：250 倍
3.维护成本：100,000 + 100x
总成本：100x + 250x + 100,000 + 100x = 450x + 100,000

In [12]:
prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need \
 help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \ 
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations 
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""
response = get_completion(prompt)
print(response)

The student's solution is almost correct, but there is a small mistake in the calculation of the maintenance cost. Let's break it down step by step:

1. **Land cost**: 
   - The cost of land is $100 per square foot.
   - For \( x \) square feet, the land cost is \( 100x \).

2. **Solar panel cost**:
   - The cost of solar panels is $250 per square foot.
   - For \( x \) square feet, the solar panel cost is \( 250x \).

3. **Maintenance cost**:
   - There is a flat annual maintenance cost of $100,000.
   - Additionally, there is a maintenance cost of $10 per square foot.
   - For \( x \) square feet, the additional maintenance cost is \( 10x \).
   - Therefore, the total maintenance cost is \( 100,000 + 10x \).

Now, let's sum up all the costs:

- Land cost: \( 100x \)
- Solar panel cost: \( 250x \)
- Maintenance cost: \( 100,000 + 10x \)

Total cost for the first year of operations:
\[
100x + 250x + 100,000 + 10x = 360x + 100,000
\]

So, the correct total cost function for the first ye

#### Note that the student's solution is actually not correct.
#### 请注意，学生的解决方案实际上是不正确的。
#### We can fix this by instructing the model to work out its own solution first.
下方提示词中文翻译，用通义千问返回内容不对，请优化提示词使得能按要求输出

您的任务是确定学生的解决方案是否正确。
要解决此问题，请执行以下操作：
- 首先，找出问题的解决方案，包括最终总数。
- 然后将您的解决方案与学生的解决方案进行比较，并评估学生的解决方案是否正确。
在您自己解决该问题之前，不要判断学生的解决方案是否正确。

使用以下格式：
问题：
```
问题在这里
```
学生的解决方案：
```
学生的解决方案在这里
```
实际解决方案：
```
制定解决方案的步骤以及您的解决方案
```
学生的解答和刚刚计算的实际解答是否相同：
```
是还是不是
```
学生成绩：
```
正确或不正确
```

问题：
```
我正在建造一个太阳能装置，需要财务方面的帮助。
- 土地成本 100 美元/平方英尺
- 我可以以每平方英尺 250 美元的价格购买太阳能电池板
- 我协商了一份维护合同，每年固定费用为 10 万美元，另外每平方英尺 10 美元
第一年运营的总成本与平方英尺数的关系是多少？
```
学生的解决方案：
```
令 x 为安装尺寸（以平方英尺为单位）。
成本：
1.土地成本：100 倍
2.太阳能电池板成本：250 倍
3.维护成本：100,000 + 100x
总成本：100x + 250x + 100,000 + 100x = 450x + 100,000
```
实际解决方案：

In [13]:
prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem including the final total. 
- Then compare your solution to the student's solution \ 
and evaluate if the student's solution is correct or not. 
Don't decide if the student's solution is correct until 
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help \
working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
``` 
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)

```
Let's break down the costs step by step:

1. **Land cost**: The cost of the land is $100 per square foot.
   - If \( x \) is the number of square feet, the land cost is \( 100x \).

2. **Solar panel cost**: The cost of the solar panels is $250 per square foot.
   - If \( x \) is the number of square feet, the solar panel cost is \( 250x \).

3. **Maintenance cost**: The maintenance cost consists of a flat fee and a variable fee.
   - Flat fee: $100,000
   - Variable fee: $10 per square foot
   - If \( x \) is the number of square feet, the maintenance cost is \( 100,000 + 10x \).

Now, let's sum up all the costs:
- Land cost: \( 100x \)
- Solar panel cost: \( 250x \)
- Maintenance cost: \( 100,000 + 10x \)

Total cost for the first year:
\[ \text{Total cost} = 100x + 250x + 100,000 + 10x \]
\[ \text{Total cost} = (100x + 250x + 10x) + 100,000 \]
\[ \text{Total cost} = 360x + 100,000 \]

So, the total cost for the first year as a function of the number of square feet \( x \) is:
\[ 

## Model Limitations: Hallucinations
## 模型限制：幻觉
- Boie is a real company, the product name is not real.
- Boie是一家真实的公司，产品名称不是真实的。

这个例子现在执行已经不会胡编乱造了，大模型一直在升级

In [14]:
prompt = f"""
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
"""
response = get_completion(prompt)
print(response)

As of my last update, there isn't a specific product called the "AeroGlide UltraSlim Smart Toothbrush by Boie." It's possible that this is a new product or a hypothetical one. However, I can provide you with some general information about smart toothbrushes and the Boie brand, which might help you understand what such a product might offer.

### Boie
Boie is a company known for creating sustainable and eco-friendly personal care products. They are particularly known for their toothbrushes and replacement heads, which are designed to be more environmentally friendly than traditional plastic toothbrushes. Boie's products often feature replaceable heads and handles made from materials that are more durable and less harmful to the environment.

### Smart Toothbrush Features
If the AeroGlide UltraSlim Smart Toothbrush by Boie were a real product, it might include the following features, which are common in other smart toothbrushes:

1. **Bluetooth Connectivity**: The toothbrush could connec

## Try experimenting on your own!
## 尝试自己尝试一下！

#### Notes on using the OpenAI API outside of this classroom
#### 关于在课堂外使用 OpenAI API 的注意事项

To install the OpenAI Python library:
安装 OpenAI Python 库：
```
!pip install openai
```

The library needs to be configured with your account's secret key, which is available on the [website](https://platform.openai.com/account/api-keys). 
该库需要使用您帐户的密钥进行配置，该密钥可在[网站](https://platform.openai.com/account/api-keys)上找到。

You can either set it as the `OPENAI_API_KEY` environment variable before using the library:
您可以在使用该库之前将其设置为“OPENAI_API_KEY”环境变量：
 ```
 !export OPENAI_API_KEY='sk-...'
 ```

Or, set `openai.api_key` to its value:
或者，将 `openai.api_key` 设置为其值：

```
import openai
openai.api_key = "sk-..."
```

#### A note about the backslash
#### 关于反斜杠的注释
- In the course, we are using a backslash `\` to make the text fit on the screen without inserting newline '\n' characters.
- 在课程中，我们使用反斜杠“\”使文本适合屏幕，而不插入换行符“\n”。
- GPT-3 isn't really affected whether you insert newline characters or not.  But when working with LLMs in general, you may consider whether newline characters in your prompt may affect the model's performance.
- 无论您是否插入换行符，GPT-3 都不会受到真正的影响。  但是，在一般情况下使用法学硕士时，您可能会考虑提示中的换行符是否会影响模型的性能。